In [0]:
df = spark.table("workspace.default.orders")

df.show(5)

+--------+-----------+----------+------------+-----------+--------+-----+---------+----------+
|order_id|customer_id|product_id|product_name|   category|quantity|price|     city|order_date|
+--------+-----------+----------+------------+-----------+--------+-----+---------+----------+
|       1|       C101|      P001|      Laptop|Electronics|       2|50000|  Chennai|2025-01-01|
|       2|       C102|      P002|       Phone|Electronics|       1|30000|Bangalore|2025-01-02|
|       3|       C103|      P003|    Keyboard|Accessories|       3| 1500|Hyderabad|2025-01-02|
|       4|       C104|      P001|      Laptop|Electronics|       1|50000|  Chennai|2025-01-03|
|       5|       C101|      P004|       Mouse|Accessories|       5|  800|   Mumbai|2025-01-04|
+--------+-----------+----------+------------+-----------+--------+-----+---------+----------+
only showing top 5 rows


In [0]:
df.printSchema()


root
 |-- order_id: long (nullable = true)
 |-- customer_id: string (nullable = true)
 |-- product_id: string (nullable = true)
 |-- product_name: string (nullable = true)
 |-- category: string (nullable = true)
 |-- quantity: long (nullable = true)
 |-- price: long (nullable = true)
 |-- city: string (nullable = true)
 |-- order_date: string (nullable = true)



In [0]:
from pyspark.sql.functions import count

duplicates = (
    df.groupBy("order_id")
      .agg(count("*").alias("cnt"))
      .filter("cnt > 1")
)

duplicates.show()

+--------+---+
|order_id|cnt|
+--------+---+
|       9|  2|
|      16|  2|
|      18|  2|
|      22|  2|
|      32|  2|
|      35|  2|
|      37|  2|
|      41|  2|
|      47|  2|
|      49|  2|
+--------+---+



In [0]:
from pyspark.sql.functions import col

df.filter(
    col("quantity").isNull() |
    col("city").isNull()
).show()

+--------+-----------+----------+------------+-----------+--------+-----+-------+----------+
|order_id|customer_id|product_id|product_name|   category|quantity|price|   city|order_date|
+--------+-----------+----------+------------+-----------+--------+-----+-------+----------+
|       9|       C108|      P008|  Headphones|Accessories|    NULL| 2500|Chennai|2025-01-06|
|      18|       C117|      P017|         RAM|Electronics|       4| 3500|   NULL|2025-01-12|
|      35|       C134|      P034|      Router| Networking|       3| 2600|   NULL|2025-01-21|
|      41|       C140|      P040|  Power Bank|Accessories|    NULL| 1300|  Delhi|2025-01-24|
+--------+-----------+----------+------------+-----------+--------+-----+-------+----------+



In [0]:
df.filter(col("quantity") < 0).show()

+--------+-----------+----------+------------+-----------+--------+-----+---------+----------+
|order_id|customer_id|product_id|product_name|   category|quantity|price|     city|order_date|
+--------+-----------+----------+------------+-----------+--------+-----+---------+----------+
|      16|       C115|      P015|         UPS|Electronics|      -2| 7000|  Chennai|2025-01-11|
|      32|       C131|      P031|      Webcam|Accessories|      -2| 2000|Hyderabad|2025-01-19|
|      47|       C146|      P046|       Mouse|Accessories|      -8|  950|   Mumbai|2025-01-27|
+--------+-----------+----------+------------+-----------+--------+-----+---------+----------+



In [0]:
from pyspark.sql.functions import try_to_date

df = df.withColumn(
    "parsed_date",
    try_to_date("order_date", "yyyy-MM-dd")
)

df.filter(col("parsed_date").isNull()).show()

+--------+-----------+----------+------------+-----------+--------+-----+-------+----------+-----------+
|order_id|customer_id|product_id|product_name|   category|quantity|price|   city|order_date|parsed_date|
+--------+-----------+----------+------------+-----------+--------+-----+-------+----------+-----------+
|      22|       C121|      P021|  Microphone|Accessories|       2| 2200|Kolkata|2025-15-14|       NULL|
|      37|       C136|      P036|         UPS|Electronics|       1| 7200|Chennai|2025-13-22|       NULL|
|      49|       C148|      P048|     Printer|Electronics|       1|16000|   Pune|2025-25-28|       NULL|
+--------+-----------+----------+------------+-----------+--------+-----+-------+----------+-----------+



In [0]:
from pyspark.sql.functions import expr, col

silver_df = (
    df.withColumn(
        "parsed_date",
        expr("try_to_timestamp(order_date, 'yyyy-MM-dd')")
    )
    .dropDuplicates()
    .filter(col("quantity") > 0)
    .filter(col("city").isNotNull())
    .filter(col("parsed_date").isNotNull())
)

silver_df = silver_df.withColumn(
    "revenue",
    col("quantity") * col("price")
)

silver_df.show()

In [0]:
from pyspark.sql.functions import col

silver_df = silver_df.withColumn(
    "revenue",
    col("quantity") * col("price")
)

In [0]:
silver_df.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("workspace.default.orders_silver")

In [0]:
from pyspark.sql.functions import sum

gold_df = (
    silver_df.groupBy("product_name")
    .agg(
        sum("revenue").alias("total_sales")
    )
)

In [0]:
gold_df.show()

+-------------+-----------+
| product_name|total_sales|
+-------------+-----------+
|       Laptop|     312000|
|        Phone|     129000|
|     Keyboard|      19300|
|        Mouse|      17000|
|      Monitor|      64000|
|      Printer|      60000|
|       Tablet|     127000|
|   Headphones|      18400|
|      Speaker|      21800|
|       Webcam|       9400|
|          SSD|      30600|
|    Hard Disk|      17100|
|       Router|      12800|
|       Switch|      13900|
|          UPS|      21200|
|Graphics Card|      93000|
|          RAM|      21200|
|    Processor|      58000|
|   Power Bank|      10100|
|  Smart Watch|      24500|
+-------------+-----------+
only showing top 20 rows


In [0]:
gold_df.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("workspace.default.orders_gold")

In [0]:
spark.table("workspace.default.orders_gold").show()

+-------------+-----------+
| product_name|total_sales|
+-------------+-----------+
|       Webcam|       9400|
|   Power Bank|      10100|
|      Speaker|      21800|
|      Printer|      60000|
|        Mouse|      17000|
|       Tablet|     127000|
|      Monitor|      64000|
|          UPS|      21200|
|          SSD|      30600|
|    Processor|      58000|
|     Keyboard|      19300|
|       Switch|      13900|
|   Microphone|      11600|
|  Smart Watch|      24500|
|   Headphones|      18400|
|       Laptop|     312000|
|       Router|      12800|
|Graphics Card|      93000|
|        Phone|     129000|
|    Hard Disk|      17100|
+-------------+-----------+
only showing top 20 rows


In [0]:
%sql
SELECT *
FROM workspace.default.orders_gold
ORDER BY total_sales DESC;

product_name,total_sales
Laptop,312000
Phone,129000
Tablet,127000
Graphics Card,93000
Monitor,64000
Printer,60000
Processor,58000
SSD,30600
Smart Watch,24500
Speaker,21800


In [0]:
from pyspark.sql.functions import sum

city_sales = (
    silver_df.groupBy("city")
    .agg(
        sum("revenue").alias("city_revenue")
    )
)

city_sales.show()

+---------+------------+
|     city|city_revenue|
+---------+------------+
|  Chennai|      351600|
|Bangalore|      243800|
|Hyderabad|       49900|
|   Mumbai|      105600|
|    Delhi|       91200|
|     Pune|       97300|
|  Kolkata|      152500|
+---------+------------+



In [0]:
city_sales.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("workspace.default.city_sales_gold")

In [0]:
category_sales = (
    silver_df.groupBy("category")
    .agg(
        sum("revenue").alias("category_revenue")
    )
)

category_sales.show()

+-----------+----------------+
|   category|category_revenue|
+-----------+----------------+
|Electronics|          957600|
|Accessories|          107600|
| Networking|           26700|
+-----------+----------------+



In [0]:
category_sales.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("workspace.default.category_sales_gold")

In [0]:
customer_sales = (
    silver_df.groupBy("customer_id")
    .agg(
        sum("revenue").alias("total_spent")
    )
)

customer_sales.show()

+-----------+-----------+
|customer_id|total_spent|
+-----------+-----------+
|       C101|     104000|
|       C102|      30000|
|       C103|       4500|
|       C104|      50000|
|       C105|      24000|
|       C106|      15000|
|       C107|      40000|
|       C108|      10000|
|       C109|       7000|
|       C110|       5400|
|       C111|      12000|
|       C112|       5500|
|       C113|       5000|
|       C114|       4500|
|       C115|      14000|
|       C116|      45000|
|       C117|      14000|
|       C118|      28000|
|       C119|       3600|
|       C120|      16000|
+-----------+-----------+
only showing top 20 rows


In [0]:
customer_sales.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("workspace.default.customer_sales_gold")

In [0]:
%sql
DESCRIBE HISTORY workspace.default.orders_silver;

version,timestamp,userId,userName,operation,operationParameters,job,notebook,queryHistoryStatementId,clusterId,readVersion,isolationLevel,isBlindAppend,operationMetrics,userMetadata,engineInfo
0,2026-09-18T05:32:51.000Z,72896342663495,sakthivelmurugan1211@gmail.com,CREATE OR REPLACE TABLE AS SELECT,"Map(isV1SaveAsTableOverwrite -> true, partitionBy -> [], clusterBy -> [], description -> null, isManaged -> true, properties -> {""delta.parquet.format.version"":""2.12.0"",""delta.parquet.format.version.afe.internal"":""2.12.0"",""delta.parquet.compression.codec"":""zstd"",""delta.enableDeletionVectors"":""true""}, statsOnLoad -> true)",null,List(617766879785972),ed551595-1a1e-41c7-832e-829645564031,0918-051926-koxwdvix-v2n,null,WriteSerializable,false,"Map(numFiles -> 1, numRemovedFiles -> 0, numRemovedBytes -> 0, numDeletionVectorsRemoved -> 0, numOutputRows -> 50, numOutputBytes -> 4658)",null,Databricks-Runtime/19.6.x-aarch64-photon-scala2.13


In [0]:
%sql
OPTIMIZE workspace.default.orders_silver;

path,metrics
,"List(0, 0, List(null, null, 0.0, 0, 0), List(null, null, 0.0, 0, 0), 0, null, null, 0, 0, 1, 1, true, 0, 0, 1789709814405, 1789709815621, 8, 0, null, List(0, 0), null, 11, 11, 0, 0, null, null, 0)"


In [0]:
%sql
VACUUM workspace.default.orders_silver RETAIN 168 HOURS;

path
""


In [0]:
%sql
SELECT
    product_name,
    SUM(revenue) AS total_sales
FROM workspace.default.orders_silver
GROUP BY product_name
ORDER BY total_sales DESC
LIMIT 5;

product_name,total_sales
Laptop,312000
Phone,129000
Tablet,127000
Graphics Card,93000
Monitor,64000


In [0]:
%sql
SELECT
    city,
    SUM(revenue) AS total_revenue
FROM workspace.default.orders_silver
GROUP BY city
ORDER BY total_revenue DESC;

city,total_revenue
Chennai,351600
Bangalore,243800
Kolkata,152500
Mumbai,105600
Pune,97300
Delhi,91200
Hyderabad,49900


In [0]:
%sql
SELECT
    customer_id,
    SUM(revenue) AS total_spent
FROM workspace.default.orders_silver
GROUP BY customer_id
ORDER BY total_spent DESC
LIMIT 10;

customer_id,total_spent
C143,110000
C101,104000
C149,66000
C123,64000
C122,52000
C104,50000
C137,48000
C116,45000
C107,40000
C144,35000


In [0]:
from pyspark.sql.functions import sum

silver_df.groupBy("product_name") \
         .agg(sum("revenue").alias("sales")) \
         .orderBy("sales", ascending=False) \
         .show()

+-------------+------+
| product_name| sales|
+-------------+------+
|       Laptop|312000|
|        Phone|129000|
|       Tablet|127000|
|Graphics Card| 93000|
|      Monitor| 64000|
|      Printer| 60000|
|    Processor| 58000|
|          SSD| 30600|
|  Smart Watch| 24500|
|      Speaker| 21800|
|          UPS| 21200|
|          RAM| 21200|
|     Keyboard| 19300|
|   Headphones| 18400|
|    Hard Disk| 17100|
|        Mouse| 17000|
|       Switch| 13900|
|       Router| 12800|
|   Microphone| 11600|
|   Power Bank| 10100|
+-------------+------+
only showing top 20 rows


In [0]:
silver_df.groupBy("customer_id") \
         .agg(sum("revenue").alias("spent")) \
         .orderBy("spent", ascending=False) \
         .show()
         

+-----------+------+
|customer_id| spent|
+-----------+------+
|       C143|110000|
|       C101|104000|
|       C149| 66000|
|       C123| 64000|
|       C122| 52000|
|       C104| 50000|
|       C137| 48000|
|       C116| 45000|
|       C107| 40000|
|       C144| 35000|
|       C102| 30000|
|       C139| 30000|
|       C127| 29000|
|       C118| 28000|
|       C147| 27000|
|       C105| 24000|
|       C128| 21000|
|       C132| 18600|
|       C120| 16000|
|       C148| 16000|
+-----------+------+
only showing top 20 rows
